In [1]:
from winnow_fcns import *
from sim_utils import *
import numpy as np
from pathlib import Path
import os
import csv
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
#------------------files---------------------------------------------
current_file = Path.cwd()

init_file = current_file.parent / "initial_cyrus_bits.csv"
# ch1_file = current_file.parent / "ch1_cyrus.csv"
# ch2_file = current_file.parent / "ch2_cyrus.csv"
ch1_file = current_file.parent / "4.16/ch1_cyrus16new.csv"
ch2_file = current_file.parent / "4.16/ch2_cyrus16new.csv"


def load_data(init_file, ch1_file, ch2_file):
    init = np.loadtxt(init_file, delimiter=',', dtype=int)
    ch1 = np.loadtxt(ch1_file, delimiter=',')
    ch2 = np.loadtxt(ch2_file, delimiter=',')
    return init, ch1, ch2


init_key, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)


def get_curr_qber(alice_winnow, bob_winnow):
    matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

    # Calculate percentage
    percentage = matches.mean() * 100

    return percentage


In [12]:
def b92_bit(ch1, ch2):
    """
    Map photon detector counts to Bob's bit according to B92 protocol.
    ch1: counts for |90°> detector (testing for 0)
    ch2: counts for |-45°> detector (testing for 1)
    Returns 0, 1, or None (inconclusive)
    """
    if ch1 > 0 and ch2 == 0:
        return 0
    elif ch2 > 0 and ch1 == 0:
        return 1
    else:
        return -1 # inconclusive, discard

In [13]:
def counts_to_bits(ch1, ch2):
    """
    Returns:
        B_received: array with {0,1,-1}
    """
    B = np.full(len(ch1), -1, dtype=int)

    for i in range(len(ch1)):
        if ch1[i] > ch2[i]:
            B[i] = 1
        elif ch2[i] > ch1[i]:
            B[i] = 0
        else:
            B[i] = -1  # tie → discard

    return B

In [15]:
def create_sifted_index_arrays(A_repeat, B_received, N_duplicate):
    A_sifted = []
    B_sifted = []
    index_array = []

    for i in range(len(B_received)):
        if B_received[i] != -1:
            A_sifted.append(A_repeat[i])
            B_sifted.append(B_received[i])
            index_array.append(i // N_duplicate)

    return np.array(A_sifted), np.array(B_sifted), np.array(index_array)


In [21]:
# --- FINAL INTEGRATED WORKFLOW (DISCARD VERSION) ---

def cleanup(alice_winnow, bob_winnow):

    def keys_are_perfect(a_winnow, b_winnow):
        return a_winnow._key_string.bits == b_winnow._key_string.bits

    # 1. Main Correction Loop (Assuming your while loop finished)

    # 2. THE CLEANUP PASS (Handles the '2-bit error' clusters safely)
    if not keys_are_perfect(alice_winnow, bob_winnow):
        print("\nResidual errors detected. Running SAFE Cleanup (Discarding)...")
        
        # Shuffle to ensure errors are independent
        alice_winnow.next_pass(permute_bits=True)
        bob_winnow.next_pass(permute_bits=True)
        
        # Use a small block size to minimize data loss
        new_block_size = 4
        h_matrix_4 = np.array([[0, 1, 1, 1], [1, 0, 1, 1]])
        
        for obj in [alice_winnow, bob_winnow]:
            obj._block_size = new_block_size
            obj._parity_check_matrix = h_matrix_4
            obj._num_of_blocks = len(obj._key_string.bits) // new_block_size

        # Find which blocks have mismatches
        bad_blocks = []
        for i in range(alice_winnow._num_of_blocks):
            if alice_winnow.get_syndrome(i) != bob_winnow.get_syndrome(i):
                bad_blocks.append(i)
        
        if bad_blocks:
            print(f"Found {len(bad_blocks)} suspicious blocks. Discarding them to guarantee 0% QBER.")
            alice_winnow.discard_syndrome_bits(bad_blocks)
            bob_winnow.discard_syndrome_bits(bad_blocks)

    # 3. FINAL SYNCHRONIZATION
    alice_winnow._key_string.unpermute_all()
    bob_winnow._key_string.unpermute_all()

    # 4. OUTPUT KEYS
    final_a = alice_winnow._key_string.bits
    final_b = bob_winnow._key_string.bits

    print("\n" + "="*40)
    print(f"Final Key Length: {len(final_a)}")
    if final_a == final_b:
        print("✅ SUCCESS: Keys are 100% identical.")
    else:
        # If this triggers, your unpermute_all logic is likely fighting the discards
        diffs = [i for i, (x, y) in enumerate(zip(final_a, final_b)) if x != y]
        print(f"❌ STILL DIFFER: {len(diffs)} bits at indices {diffs}")
    print("="*40)

In [ ]:
# def winnow_pipeline(init_file, ch1_file, ch2_file, block_schedule, current_csv, N_duplicate=16, rng=None, seed=None):

#     A_repeat, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)

#     B_received = counts_to_bits(ch1, ch2)

#     A_sifted, B_sifted, index_array = create_sifted_index_arrays(
#         A_repeat, B_received, N_duplicate
#     )


    
#     alice_key = MockBitBuffer(list(A_sifted))
#     bob_key   = MockBitBuffer(list(B_sifted))

#     # Initialize Alice and Bob
#     # Note: Pass internal copy of schedule to avoid modifying the global list
#     alice_winnow = Winnow(raw_key=alice_key, perm_seed=seed, rng=rng, block_size_schedule=list(block_schedule))
#     bob_winnow = Winnow(raw_key=bob_key, perm_seed=seed, rng=rng, block_size_schedule=list(block_schedule))

#     # 1. Initialize CSV with headers
#     with open(current_csv, 'w', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow(["Pass_Number", "QBER", "Block_Size"]) 

#     def log_progress():
#         qber = get_curr_qber(alice_winnow, bob_winnow)
#         pass_num = alice_winnow._pass_number
#         block_size = alice_winnow._block_size
#         print(f"At pass {pass_num} and size {block_size} Qber {qber}%")
        
#         with open(current_csv, 'a', newline='') as f:
#             writer = csv.writer(f)
#             writer.writerow([pass_num, qber, block_size])
#         return qber

#     # --- START RECONCILIATION ---

#     log_progress()

#     # 2. First Pass Setup
#     alice_winnow.first_pass()
#     bob_winnow.first_pass()

#     # 3. Perform First Pass Corrections
#     for i in range(alice_winnow._num_of_blocks):
#         alice_syndrome = alice_winnow.get_syndrome(i)
#         bob_winnow.fix_with_syndrome(i, alice_syndrome)

#     # 4. Main Loop
#     while True:
#         # Check if we should stop before starting a new pass
#         current_qber = get_curr_qber(alice_winnow, bob_winnow)
#         if (alice_winnow.get_num_remaining_passes() <= 0) or (current_qber < 1e-9):
#             break

#         # A. DISCARD parity bits from the pass we JUST finished
#         # Information was leaked during the syndrome exchange
#         prev_block_size = alice_winnow._block_size
#         prev_num_blocks = alice_winnow._num_of_blocks
#         alice_winnow._key_string.discard_parity_bits(prev_block_size, prev_num_blocks)
#         bob_winnow._key_string.discard_parity_bits(prev_block_size, prev_num_blocks)

#         log_progress()

#         # B. PREPARE next pass
#         status_a = alice_winnow.next_pass(permute_bits=True) 
#         status_b = bob_winnow.next_pass(permute_bits=True) 

#         if status_a == -1 or status_b == -1:
#             print("Reconciliation complete: Schedule exhausted.")
#             break


#         # D. PERFORM corrections for this new pass
#         for i in range(alice_winnow._num_of_blocks):
#             alice_syndrome = alice_winnow.get_syndrome(i)
#             bob_winnow.fix_with_syndrome(i, alice_syndrome)
    
#     # 5. Final log after all loops finish
#     log_progress()
#     print(f"Final Remaining Passes: {alice_winnow.get_num_remaining_passes()}")
#     print(f"Final QBER: {get_curr_qber(alice_winnow, bob_winnow)}%")

#     return None


# many_block_schedules = [
#     [4,8,8,16,64],
#     [8,16,64],
#     # [8,16,64,0,0,0,0,0],

#     # [4,2,2,0,0,0,0,0],
#     # [4,2,1,0,0,0,0,0],
#     # # # [8,4,1,0,0,0,0,0],
#     # [2,2,0,0,0,0,0,0],
#     # # [1,1,0,0,0,0,0,0]
# ]

# output_file_names = [
#     'qber16_pass_4_8_8_16_64.csv',
#     'qber16_pass_8_16_64.csv',
#     # 'qber8_pass_42200000.csv',
#     # 'qber8_pass_42100000.csv',
#     # # 'qber_pass_41000000.csv',
#     # 'qber8_pass_22000000.csv',
#     # # 'qber4_pass_11000000.csv'
# ]

# # # qber_for_schedule = []
# # # num_pass_for_schedule = []
# # rng = np.random.default_rng() 
# seed_schedules = [
#     23,
#     45,
#     # 95,
#     # 56,
#     # 82,
# ]



# for block_schedule, output_file, seed in zip(many_block_schedules, output_file_names, seed_schedules):
#     winnow_pipeline(init_file, ch1_file, ch2_file, block_schedule, output_file, N_duplicate=8, seed=seed)
#     # qber_for_schedule.append(qber_list)
#     # num_pass_for_schedule.append(num_pass)




At pass 0 and size None Qber 1.3671169686569502%
The number of remaining passes is 4
At pass 1 and size 4 Qber 0.07619806956480053%
The number of remaining passes is 3
At pass 2 and size 8 Qber 0.0007208914832438386%
The number of remaining passes is 2
At pass 3 and size 8 Qber 0.0%
The number of remaining passes is 1
At pass 4 and size 16 Qber 0.0%
The number of remaining passes is 1
Final Remaining Passes: 1
Final QBER: 0.0%
At pass 0 and size None Qber 1.3671169686569502%
The number of remaining passes is 2
At pass 1 and size 8 Qber 0.18880132067208658%
The number of remaining passes is 1
At pass 2 and size 16 Qber 0.007151260698055318%
The number of remaining passes is 0
At pass 3 and size 64 Qber 0.00026913346713111413%
The number of remaining passes is 0
Final Remaining Passes: 0
Final QBER: 0.00026913346713111413%


In [ ]:
def winnow_pipeline(init_file, ch1_file, ch2_file, block_schedule, current_csv, N_duplicate=16, rng=None, seed=None):

    A_repeat, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)

    B_received = counts_to_bits(ch1, ch2)

    A_sifted, B_sifted, index_array = create_sifted_index_arrays(
        A_repeat, B_received, N_duplicate
    )


    
    alice_key = MockBitBuffer(list(A_sifted))
    bob_key   = MockBitBuffer(list(B_sifted))

    # Initialize Alice and Bob
    # Note: Pass internal copy of schedule to avoid modifying the global list
    alice_winnow = Winnow(raw_key=alice_key, perm_seed=seed, rng=rng, block_size_schedule=list(block_schedule))
    bob_winnow = Winnow(raw_key=bob_key, perm_seed=seed, rng=rng, block_size_schedule=list(block_schedule))

    # 1. Initialize CSV with headers
    with open(current_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Pass_Number", "QBER", "Block_Size", "Net Exposed Bits"]) 

    def log_progress():
        qber = get_curr_qber(alice_winnow, bob_winnow)
        pass_num = alice_winnow._pass_number
        block_size = alice_winnow._block_size
        net_bits_exposed = alice_winnow._net_bits_exposed
        print(f"At pass {pass_num} and size {block_size} Qber {qber}%")
        
        with open(current_csv, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([pass_num, qber, block_size, net_bits_exposed])
        return qber

    # --- START RECONCILIATION ---

    log_progress()

    # 2. First Pass Setup
    alice_winnow.first_pass()
    bob_winnow.first_pass()

    # 3. Perform First Pass Corrections
    for i in range(alice_winnow._num_of_blocks):
        alice_syndrome = alice_winnow.get_syndrome(i)
        bob_winnow.fix_with_syndrome(i, alice_syndrome)

    # 4. Main Loop
    while True:
        # Check if we should stop before starting a new pass
        current_qber = get_curr_qber(alice_winnow, bob_winnow)
        if (alice_winnow.get_num_remaining_passes() <= 0) or (current_qber < 1e-9):
            break

        # A. DISCARD parity bits from the pass we JUST finished
        # Information was leaked during the syndrome exchange
        prev_block_size = alice_winnow._block_size
        prev_num_blocks = alice_winnow._num_of_blocks
        alice_winnow._key_string.discard_parity_bits(prev_block_size, prev_num_blocks)
        bob_winnow._key_string.discard_parity_bits(prev_block_size, prev_num_blocks)

        log_progress()

        # B. PREPARE next pass
        status_a = alice_winnow.next_pass(permute_bits=True) 
        status_b = bob_winnow.next_pass(permute_bits=True) 

        if status_a == -1 or status_b == -1:
            print("Reconciliation complete: Schedule exhausted.")
            break


        # D. PERFORM corrections for this new pass
        for i in range(alice_winnow._num_of_blocks):
            alice_syndrome = alice_winnow.get_syndrome(i)
            bob_winnow.fix_with_syndrome(i, alice_syndrome)
    
    # 5. Final log after all loops finish
    log_progress()
    print(f"Final Remaining Passes: {alice_winnow.get_num_remaining_passes()}")
    print(f"Final QBER: {get_curr_qber(alice_winnow, bob_winnow)}%")

    cleanup(alice_winnow, bob_winnow)

    return None


many_block_schedules = [
    [4,8,8,16,64],
    [8,16,64],
    [8,8,16,64],
    [16,16,64,64],
    [4,4,4,8]
    # [8,16,64,0,0,0,0,0],

    # [4,2,2,0,0,0,0,0],
    # [4,2,1,0,0,0,0,0],
    # # # [8,4,1,0,0,0,0,0],
    # [2,2,0,0,0,0,0,0],
    # # [1,1,0,0,0,0,0,0]
]

output_file_names = [
    'qber16_pass_4_8_8_16_64.csv',
    'qber16_pass_8_16_64.csv',
    'qber16_pass_8_8_16_64.csv',
    'qber16_pass_16_16_64_64.csv',
    'qber16_pass_4_4_4_8.csv',
    # 'qber8_pass_42200000.csv',
    # 'qber8_pass_42100000.csv',
    # # 'qber_pass_41000000.csv',
    # 'qber8_pass_22000000.csv',
    # # 'qber4_pass_11000000.csv'
]


# # qber_for_schedule = []
# # num_pass_for_schedule = []
# rng = np.random.default_rng() 
seed_schedules = [
    74,
    49,
    95,
    56,
    82,
]



for block_schedule, output_file, seed in zip(many_block_schedules, output_file_names, seed_schedules):
    winnow_pipeline(init_file, ch1_file, ch2_file, block_schedule, output_file, N_duplicate=8, seed=seed)
    # qber_for_schedule.append(qber_list)
    # num_pass_for_schedule.append(num_pass)




At pass 0 and size None Qber 1.3671169686569502%
The number of remaining passes is 4
At pass 1 and size 4 Qber 0.07619806956480053%
The number of remaining passes is 3
At pass 2 and size 8 Qber 0.000576713186595071%
The number of remaining passes is 2
At pass 3 and size 8 Qber 0.0%
The number of remaining passes is 1
At pass 4 and size 16 Qber 0.0%
The number of remaining passes is 1
Final Remaining Passes: 1
Final QBER: 0.0%

Final Key Length: 1820658
✅ SUCCESS: Keys are 100% identical.
At pass 0 and size None Qber 1.3671169686569502%
The number of remaining passes is 2
At pass 1 and size 8 Qber 0.18880132067208658%
The number of remaining passes is 1
At pass 2 and size 16 Qber 0.00576714572423816%
The number of remaining passes is 0
At pass 3 and size 64 Qber 3.8447638161587735e-05%
The number of remaining passes is 0
Final Remaining Passes: 0
Final QBER: 3.8447638161587735e-05%

Residual errors detected. Running SAFE Cleanup (Discarding)...
Time to terminate: End of schedule/total p

In [ ]:
# def winnow_pipeline(ch1, ch2, block_schedule, current_csv, rng=None, seed=None):
#     alice_key = MockBitBuffer(list(ch1))
#     bob_key   = MockBitBuffer(list(ch2))


#     # Initialize Alice and Bob
#     alice_winnow = Winnow(raw_key=alice_key, perm_seed=seed, rng=rng, block_size_schedule=block_schedule)
#     bob_winnow = Winnow(raw_key=bob_key, perm_seed=seed, rng=rng, block_size_schedule=block_schedule)

#     # 1. Initialize CSV with headers (overwrites old file)
#     with open(current_csv, 'w', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow(["Pass_Number", "QBER", "Block_Size"]) 

#     # Helper function to log a single row
#     def log_progress():
#         qber = get_curr_qber(alice_winnow, bob_winnow)
#         pass_num = alice_winnow._pass_number
#         block_size = alice_winnow._block_size
#         print(f"At pass {pass_num} and size {block_size} Qber {qber}%")
        
#         with open(current_csv, 'a', newline='') as f:
#             writer = csv.writer(f)
#             writer.writerow([pass_num, qber, block_size])
#         return qber, pass_num

#     # Log initial QBER (Pass 0)
#     log_progress()

#     # print(alice_winnow._block_size)

#     # First Pass
#     alice_winnow.first_pass()
#     bob_winnow.first_pass()
#     print(alice_winnow.get_block_size_schedule())
#     print(alice_winnow._block_size)
    
#     # Fix first pass blocks
#     for i in range(alice_winnow._num_of_blocks):
#         alice_syndrome = alice_winnow.get_syndrome(i)
#         bob_winnow.fix_with_syndrome(i, alice_syndrome)

#     # Main Loop
#     while (alice_winnow.get_num_remaining_passes() > 0) and (get_curr_qber(alice_winnow, bob_winnow) > 1e-9):
#         # 2. Discard parity bits from previous pass
#         block_size = alice_winnow._block_size
#         num_blocks = alice_winnow._num_of_blocks
#         alice_winnow._key_string.discard_parity_bits(block_size, num_blocks)
#         bob_winnow._key_string.discard_parity_bits(block_size, num_blocks)

#         # 3. Log results BEFORE the next shuffle/pass
#         log_progress()

#         # 4. Prepare next pass
#         status = alice_winnow.next_pass(permute_bits=True) 
#         status = bob_winnow.next_pass(permute_bits=True) 

#         if status == -1:
#             print("Reconciliation complete: Schedule exhausted.")
#         break

#         print(alice_winnow._block_size)


#         # 5. Correct errors for current pass
#         for i in range(alice_winnow._num_of_blocks):
#             alice_syndrome = alice_winnow.get_syndrome(i)
#             bob_winnow.fix_with_syndrome(i, alice_syndrome)
    
#     # Final log after last correction
#     log_progress()

#     print(alice_winnow.get_num_remaining_passes())
#     print(get_curr_qber(alice_winnow, bob_winnow))

#     return None # Lists are now saved to disk pass-by-pass


# many_block_schedules = [
#     [4,8,8,16,64,0,0,0],
#     # [8,16,64,0,0,0,0,0],

#     # [4,2,2,0,0,0,0,0],
#     # [4,2,1,0,0,0,0,0],
#     # # # [8,4,1,0,0,0,0,0],
#     # [2,2,0,0,0,0,0,0],
#     # # [1,1,0,0,0,0,0,0]
# ]

# output_file_names = [
#     'qber8_pass_4_8_8_16_64.csv',
#     # 'qber8_pass_42200000.csv',
#     # 'qber8_pass_42100000.csv',
#     # # 'qber_pass_41000000.csv',
#     # 'qber8_pass_22000000.csv',
#     # # 'qber4_pass_11000000.csv'
# ]

# # # qber_for_schedule = []
# # # num_pass_for_schedule = []
# # rng = np.random.default_rng() 
# seed_schedules = [
#     23,
#     # 45,
#     # 95,
#     # 56,
#     # 82,
# ]



# for block_schedule, output_file, seed in zip(many_block_schedules, output_file_names, seed_schedules):
#     winnow_pipeline(ch1, ch2, block_schedule, output_file, seed=seed)
#     # qber_for_schedule.append(qber_list)
#     # num_pass_for_schedule.append(num_pass)




At pass 0 and size None Qber 23.77789306640625%
[4, 8, 8, 16, 64, 0, 0, 0]
4
The number of remaining passes is 7
At pass 1 and size 4 Qber 16.5343017578125%
At pass 2 and size 8 Qber 16.5343017578125%


In [5]:

many_block_schedules = [
    [2,2,4,4,4,4,4,4],
    # [4,2,2,0,0,0,0,0],
    # # [4,2,1,0,0,0,0,0],
    # # [8,4,1,0,0,0,0,0],
    # [2,2,0,0,0,0,0,0],
    # [1,1,0,0,0,0,0,0]
]

output_file_names = [
    'qber_pass_22444444.csv',
    # 'qber_pass_42200000.csv',
    # 'qber_pass_42100000.csv',
    # 'qber_pass_41000000.csv',
    # 'qber_pass_22000000.csv',
    # 'qber_pass_11000000.csv'
]

# # qber_for_schedule = []
# # num_pass_for_schedule = []
# rng = np.random.default_rng() 
seed_schedules = [
    84,
    # 45,
    # 47,
    # 56
]

current_file = Path.cwd()

# init_file = current_file.parent / "initial_cyrus_bits.csv"
ch1_file = current_file.parent / "4.16/ch1_cyrus8new.csv"
ch2_file = current_file.parent / "ch2_cyrus.csv"



for ch1, ch2 in zip(ch1_list, ch2_list):
    for block_schedule, output_file, seed in zip(many_block_schedules, output_file_names, seed_schedules):
        winnow_pipeline(ch1, ch2, block_schedule, output_file, seed=seed)
    # qber_for_schedule.append(qber_list)
    # num_pass_for_schedule.append(num_pass)




NameError: name 'ch1_list' is not defined